# Dia 02 - Armazenamento e Arquiteturas 
- Objetivo: Compreender o impacto dos formatos de armazenamento e como as arquiteturas evoluíram do DW e LH.

## SESSÕES
- Comparativo entre diferentes formatos de dados (.csv vs .parquet)
https://www.alura.com.br/artigos/arquivos-parquet
    - Apresentar uma comparação entre o tamanho dos arquivos em disco
    - Apresentar a comparação entre o tempo de leitura entre CSV e parquet

- Simulação de um Data Lake em pastas (medallion architecture / zones)
    - Demonstrar a organização dos arquivos em diretórios /raw (dados brutos) /silver (dados limpos em parquet) e /gold (dados agregados para negócio)

- Armazenamento colunar com DuckDB
    - Apresentação do conceito de armazenamento colunar
    - Apresentar o conceito de um banco OLAP colunar-in-process
    - bancos olap vs oltp

- Staging e Padrão ELT
    - Explicar brevemente as diferenças entre etl e elt
    - Demonstrar como criar schemas e realizar carga de dados brutos direto para tabelas de staging utilizando o duckDB
    - Exibir como os schemas ingeridos ficam dispostos em relação a estrutura e disponibilidade dos dados, bem como discutir potenciais inconsistências


## 1. Comparativo entre diferentes formatos de dados: CSV e Parquet

Para ilustrar a diferença teórica entre um formato orientado a linhas (CSV) e um formato colunar com compressão nativa (Parquet), vamos colocar ambos à prova utilizando um grande volume de dados.

O código a seguir criará, em tempo de execução, um dataset fictício contendo 1 milhão de registros de vendas. Com esse volume gerado, faremos duas medições:

- Espaço em Disco: Vamos salvar o mesmo conjunto de dados nos formatos .csv e .parquet e comparar o tamanho (em MB) dos arquivos gerados.

- Desempenho de Leitura (I/O): Usaremos a biblioteca time para cronometrar os milissegundos que o Python leva para carregar cada um desses arquivos de volta para a memória.


In [2]:
import pandas as pd
import numpy as np
import time
import os

#Vamos gerar um dataset fictício com 1 MILHÃO de linhas... Isso pode levar alguns segundos hihihihihi

# 1. Criando um conjunto de dados grande para o teste
numero_linhas = 1_000_000
dados = {#observe que todas as colunas serão geradas de maneira aleatória por meio na biblioteca numpy -> np.random
    'id_venda': range(numero_linhas),
    'categoria': np.random.choice(['Eletrônicos', 'Roupas', 'Alimentos', 'Livros', 'Móveis'], numero_linhas),
    'preco': np.random.uniform(10.0, 1500.0, numero_linhas),
    'quantidade': np.random.randint(1, 10, numero_linhas),
    'status': np.random.choice(['Concluído', 'Pendente', 'Cancelado'], numero_linhas)
}

df_vendas = pd.DataFrame(dados)


# 2. Salvando em CSV
inicio_csv = time.time()#aqui vamos usar biblioteca time para conseguir marcar o inicio e o fim do processo para então compara-los.
df_vendas.to_csv("vendas_armazenamento.csv", index=False)
tempo_escrita_csv = time.time() - inicio_csv #calculo do intervalo de tempo para salvar o csv

# 3. Salvando em Parquet
inicio_parquet = time.time()
df_vendas.to_parquet("vendas_armazenamento.parquet", engine='pyarrow')
tempo_escrita_parquet = time.time() - inicio_parquet#calculo do intervalo de tempo para salvar o parquet

# 4. Comparando o Tamanho
# os.path.getsize retorna o tamanho em bytes. Dividimos por (1024*1024) para transformar em Megabytes3
tamanho_csv_mb = os.path.getsize("vendas_armazenamento.csv") / (1024 * 1024)
tamanho_parquet_mb = os.path.getsize("vendas_armazenamento.parquet") / (1024 * 1024)

# 5. Comparando o Tempo de Leitura
inicio_leitura = time.time()
pd.read_csv("vendas_armazenamento.csv")
tempo_leitura_csv = time.time() - inicio_leitura

inicio_leitura = time.time()
pd.read_parquet("vendas_armazenamento.parquet", engine='pyarrow')
tempo_leitura_parquet = time.time() - inicio_leitura

# 6. Exibindo o Grande Resultado

print("RESULTADO DO COMPARATIVO: CSV vs PARQUET")
print("-" * 50)
print(f"TAMANHO NO DISCO:")
print(f"   CSV:     {tamanho_csv_mb:.2f} MB")
print(f"   Parquet: {tamanho_parquet_mb:.2f} MB (Aproximadamente {tamanho_csv_mb/tamanho_parquet_mb:.1f}x menor!)")
print("")
print(f"TEMPO DE LEITURA (Carregar na memória):")
print(f"   CSV:     {tempo_leitura_csv:.3f} segundos")
print(f"   Parquet: {tempo_leitura_parquet:.3f} segundos (Aproximadamente {tempo_leitura_csv/tempo_leitura_parquet:.1f}x mais rápido!)")
print("-" * 50)

RESULTADO DO COMPARATIVO: CSV vs PARQUET
--------------------------------------------------
TAMANHO NO DISCO:
   CSV:     45.06 MB
   Parquet: 13.06 MB (Aproximadamente 3.4x menor!)

TEMPO DE LEITURA (Carregar na memória):
   CSV:     0.549 segundos
   Parquet: 0.174 segundos (Aproximadamente 3.2x mais rápido!)
--------------------------------------------------
